# Test embeddings subjects 1, 2

In [1]:
from mvlearn.embed import MCCA # pip install mvlearn
import numpy as np
import pandas as pd
import torch

ModuleNotFoundError: No module named 'mvlearn'

## Aux functions

In [ ]:
def evaluate_retrieval(Zx: torch.Tensor, Zy: torch.Tensor, device: str = "cpu"):
    """
    Compute retrieval metrics between two sets of embeddings.
    Assumes both Zx and Zy are of shape (n_samples, n_features) and correspond to the 
    same samples in the same order.

    Returns:
        accuracy: top-1 retrieval accuracy
        cosine: mean diagonal cosine similarity
        mean_rank: mean rank of the correct match (1 = best)
        similarities (optional): full similarity matrix
    """

    if not torch.is_tensor(Zx):
        Zx = torch.from_numpy(Zx).to(device)
    if not torch.is_tensor(Zy):
        Zy = torch.from_numpy(Zy).to(device)

    x_true_norm = torch.nn.functional.normalize(Zx, dim=1)
    x_pred_norm = torch.nn.functional.normalize(Zy, dim=1)

    # Mean cosine similarity of the diagonal
    cosine = torch.sum(x_true_norm * x_pred_norm, dim=1).mean().item()

    # Similarity matrix
    similarities = x_pred_norm @ x_true_norm.T  # (n, n)
    device = similarities.device
    target = torch.arange(similarities.shape[0], device=device)

    # Recall@1 
    top1 = torch.argmax(similarities, dim=1)
    recall = (top1 == target).float().mean().item()

    # Mean Rank
    sorted_indices = torch.argsort(similarities, dim=1, descending=True)
    ranks = (sorted_indices == target[:, None]).nonzero(as_tuple=False)[:, 1] + 1
    mean_rank = ranks.float().mean().item()

    return recall, cosine, mean_rank, top1

def get_triplet_indices(labels: torch.Tensor, repetitions: torch.Tensor) -> pd.DataFrame:
    """
    Get a DataFrame with the trial index, stimulus label, and repetition number 
    for each sample in the set.
    """
    df = pd.DataFrame({
        "trial_index": np.arange(len(labels)),
        "stimulus_label": labels.numpy().astype(int),
        "repetition": repetitions.numpy().astype(int) + 1
    })
    # Transform into a pd.dataframe with label as index, repetition as columns, and index as values
    df = df.pivot(index="stimulus_label", columns="repetition", values="trial_index").reset_index()
    df = df.rename(columns={1: "repetition_1", 2: "repetition_2", 3: "repetition_3"})
    df = df.sort_values("stimulus_label").reset_index(drop=True)

    return df

def _get_mcca_matrices(mcca, views=(0, 1, 2)):
    """
    Extract (W_v, mu_v) from an mvlearn MCCA object for the given views.
    Returns:
        Ws: list of (d, k) numpy arrays
        mus: list of (d,) numpy arrays (zeros if means_[v] is None)
    """
    Ws = []
    mus = []
    for v in views:
        W = np.asarray(mcca.loadings_[v])
        mu = mcca.means_[v]
        if mu is None:
            mu = np.zeros(W.shape[0], dtype=W.dtype)
        else:
            mu = np.asarray(mu, dtype=W.dtype)
        Ws.append(W)
        mus.append(mu)
    return Ws, mus


def distill_mcca_projector(
    X_train: np.ndarray,
    mcca,
    views: tuple[int, ...] = (0, 1, 2),
    target: str = "avg_embeddings",  # "avg_embeddings" | "single_view" | "avg_projector"
    view_for_single: int = 0,
    lam_ridge: float = 1e-2,
    fit_intercept: bool = True,
):
    """
    Distill MCCA into a single linear projector from X -> Z_target (ridge regression).

    Parameters
    ----------
    X_train : (n_samples, d)
        Training inputs in the SAME space as MCCA loadings (e.g., PCA space).
    mcca : fitted mvlearn.embed.MCCA
    views : which MCCA views to use.
    target : how to define the target latent:
        - "avg_embeddings": Z = mean_v ( (X - mu_v) @ W_v )   (recommended)
        - "single_view":    Z = (X - mu_view) @ W_view        (baseline)
        - "avg_projector":  Z = (X - mu_bar) @ W_bar          (explicit)
    view_for_single : used if target == "single_view"
    lam_ridge : ridge penalty (lambda)
    fit_intercept : if True, learns an intercept b so Z ≈ X W + b

    Returns
    -------
    W_star : (d, k)
    b_star : (k,)  (zeros if fit_intercept=False)
    Z_target : (n_samples, k)  (returned for diagnostics)
    """
    # --- extract matrices ---
    Ws, mus = _get_mcca_matrices(mcca, views=views)

    # --- build target Z ---
    if target == "avg_embeddings":
        Zs = [(X_train - mu[None, :]) @ W for W, mu in zip(Ws, mus)]
        Z_target = np.mean(np.stack(Zs, axis=0), axis=0)
    elif target == "single_view":
        # find index of view_for_single in `views` if present, else extract directly
        if view_for_single in views:
            idx = list(views).index(view_for_single)
            W, mu = Ws[idx], mus[idx]
        else:
            W, mu = (
                _get_mcca_matrices(mcca, views=(view_for_single,))[0][0],
                _get_mcca_matrices(mcca, views=(view_for_single,))[1][0],
            )
        Z_target = (X_train - mu[None, :]) @ W
    elif target == "avg_projector":
        W_bar = np.mean(np.stack(Ws, axis=0), axis=0)
        mu_bar = np.mean(np.stack(mus, axis=0), axis=0)
        Z_target = (X_train - mu_bar[None, :]) @ W_bar
    else:
        raise ValueError(
            "target must be one of: 'avg_embeddings', 'single_view', 'avg_projector'"
        )

    # --- ridge fit: Z ≈ X W + b ---
    X = X_train.astype(np.float64, copy=False)
    Z = Z_target.astype(np.float64, copy=False)

    if fit_intercept:
        X_mean = X.mean(axis=0, keepdims=True)
        Z_mean = Z.mean(axis=0, keepdims=True)
        Xc = X - X_mean
        Zc = Z - Z_mean
    else:
        X_mean = None
        Z_mean = None
        Xc = X
        Zc = Z

    d = Xc.shape[1]
    XtX = Xc.T @ Xc
    W_star = np.linalg.solve(XtX + lam_ridge * np.eye(d), Xc.T @ Zc)  # (d, k)

    if fit_intercept:
        b_star = (Z_mean - X_mean @ W_star).ravel()  # (k,)
    else:
        b_star = np.zeros(Z.shape[1], dtype=W_star.dtype)

    return W_star.astype(np.float32), b_star.astype(np.float32)

## Averaged embeddings


Each embedding is the average (on the embedding space) of the 3 repetitions

### PT format

Each checkpoint has the keys:

- **Z_train (9000, 128)**: with embeddings of the 9000 unique images (averaged the 3 repetitions)
- **labels_train (9000,)**: with the (non-contiguous) labels of the images.
- **Z_test (1000, 128)** with the embeddings of the 1000 unique images (averaged repetitions) shared between al subjects
- **labels_test (1000,)**: the ids of the test images. In both subjects are sorted in the same order and have the same images

In [ ]:
embeddings_template = "avg_ws_mlp_v1_128_768_sub-{subject:02d}.pt"


embeddings_p = torch.load(embeddings_template.format(subject=1),  map_location="cpu")
embeddings_q = torch.load(embeddings_template.format(subject=2), map_location="cpu")


for key in embeddings_p.keys():
    print(f"- {key}: {embeddings_p[key].shape} ({embeddings_p[key].dtype})")

# Test embeddings contain the same samples in the same order
assert (embeddings_p["labels_test"] == embeddings_q["labels_test"]).all(), "Test labels must match"

# Non-overlapping train ids between subjects
assert set(embeddings_p["labels_train"]).isdisjoint(set(embeddings_q["labels_train"])), "Train IDs are non-overlapping between subjects"

- Z_train: torch.Size([9000, 128]) (torch.float32)
- labels_train: torch.Size([9000]) (torch.int64)
- Z_test: torch.Size([1000, 128]) (torch.float32)
- labels_test: torch.Size([1000]) (torch.int64)


## Supervised baseline

Small test for checking that the embeddings are correct

For the supervised baseline, we need paired samples. Train sets do not overlap. We will use 500 samples for training a linear projection and rest (500) for evaluation

In [ ]:
# Shuffle the indexes to create a random train/test split
n_test = 500 
torch.manual_seed(42)
random_indices = torch.randperm(embeddings_p["labels_test"].shape[0])
test_indices = random_indices[:n_test]
train_indices = random_indices[n_test:]

Zx_supervised_train = embeddings_p["Z_test"][train_indices] # 500 x 128
Zy_supervised_train = embeddings_q["Z_test"][train_indices] # 500 x 128
Zx_supervised_test = embeddings_p["Z_test"][test_indices] # 500 x 128
Zy_supervised_test = embeddings_q["Z_test"][test_indices] # 500 x 128


# Fit MCCA on the supervised training data
mcca = MCCA(n_components=64)
mcca.fit([Zx_supervised_train, Zy_supervised_train])

# Transform both train and test data
Zx_mcca_train = mcca.transform_view(Zx_supervised_train, view=0)
Zy_mcca_train = mcca.transform_view(Zy_supervised_train, view=1)
                                    
Zx_mcca_test = mcca.transform_view(Zx_supervised_test, view=0)
Zy_mcca_test = mcca.transform_view(Zy_supervised_test, view=1)


# Train set evaluation
recall, cosine, mean_rank, paired_index_train = evaluate_retrieval(Zx_mcca_train, Zy_mcca_train, device="cuda")
print(f"Train set - R@1: {recall:.4f}, Cosine: {cosine:.4f}, Mean Rank: {mean_rank:.2f}")

# Test set evaluation
recall, cosine, mean_rank, paired_index_test = evaluate_retrieval(Zx_mcca_test, Zy_mcca_test, device="cuda")
print(f"Test set - R@1: {recall:.4f}, Cosine: {cosine:.4f}, Mean Rank: {mean_rank:.2f}")

Train set - R@1: 1.0000, Cosine: 0.8723, Mean Rank: 1.00
Test set - R@1: 0.9940, Cosine: 0.6465, Mean Rank: 1.01


## Single-trial embeddings

Now we have 27K train trials, with 9K images x 3 repetitions
And 3K test trials (1K shared images)

The test sets are sorted in the same order. Stimuli and repetition match

- **Z_train (27K, 128)**: Embeddings for training
- **labels_train (27K)**: Stimulus ids
- **repetitions_train (27K)**: Repetition of each stimuli
- **Z_test (3K, 128)**: Embeddings for testing
- **labels_test (3K)**: Stimulus ids
- **repetitions_test (3K)**: Repetition of each stimuli


In [ ]:
embeddings_template = "trial_ws_mlp_v1_128_768_sub-{subject:02d}.pt"


embeddings_p = torch.load(embeddings_template.format(subject=1), map_location="cpu")
embeddings_q = torch.load(embeddings_template.format(subject=2), map_location="cpu")


for key in embeddings_p.keys():
    print(f"- {key}: {embeddings_p[key].shape} ({embeddings_p[key].dtype})")

# Test embeddings contain the same samples in the same order
assert (embeddings_p["labels_test"] == embeddings_q["labels_test"]).all(), "Test labels must match"
assert (embeddings_p["repetitions_test"] == embeddings_q["repetitions_test"]).all(), "Repetitions must match"

# Non-overlapping train ids between subjects
assert set(embeddings_p["labels_train"]).isdisjoint(set(embeddings_q["labels_train"])), "Train IDs are non-overlapping between subjects"

- Z_train: torch.Size([27000, 128]) (torch.float32)
- labels_train: torch.Size([27000]) (torch.float32)
- repetitions_train: torch.Size([27000]) (torch.int64)
- Z_test: torch.Size([3000, 128]) (torch.float32)
- labels_test: torch.Size([3000]) (torch.float32)
- repetitions_test: torch.Size([3000]) (torch.int64)


## Supervised test - single trial

As in the averaged embeddings, we need to split the test set into two, for having paired samples.

We will take 1500 trials (500 images) for supervised train and other 500 for supervised test

In [ ]:
n_test = 500

# In the test-set, both dataframes should be identicall thus sets are already sorted
df_indexes_p = get_triplet_indices(embeddings_p["labels_test"], embeddings_p["repetitions_test"])
df_indexes_q = get_triplet_indices(embeddings_q["labels_test"], embeddings_q["repetitions_test"])
assert (df_indexes_p["stimulus_label"] == df_indexes_q["stimulus_label"]).all(), "Stimulus labels must match between subjects"

# Sample 500 stimulus_labels from the test set
np.random.seed(42)
unique_labels = df_indexes_p["stimulus_label"].unique()
supervised_test_images = np.random.choice(unique_labels, size=n_test, replace=False)

df_indexes_p["test"] = df_indexes_p["stimulus_label"].isin(supervised_test_images)
df_indexes_q ["test"] = df_indexes_q["stimulus_label"].isin(supervised_test_images)

display(df_indexes_p)

repetition,stimulus_label,repetition_1,repetition_2,repetition_3,test
0,2950,234,914,2750,True
1,2990,1804,1823,2765,False
2,3049,590,601,621,True
3,3077,407,428,431,True
4,3146,764,803,2646,False
...,...,...,...,...,...
995,72312,265,297,318,True
996,72510,1816,1862,2500,False
997,72605,636,657,658,True
998,72719,269,1131,2092,True


Train supervised MCCA on trials. We will use a 6-view approach for each repetition an each subject, and then evaluate each repetition against the other repetition

In [ ]:
n_components = 128

Zx_test = embeddings_p["Z_test"]
Zy_test = embeddings_q["Z_test"]

df_indexes_p_train = df_indexes_p.query("not test")
Zx_supervised_train_1 = Zx_test[df_indexes_p_train.repetition_1.values]  # 500 x 128
Zx_supervised_train_2 = Zx_test[df_indexes_p_train.repetition_2.values]  # 500 x 128
Zx_supervised_train_3 = Zx_test[df_indexes_p_train.repetition_3.values]  # 500 x 128

df_indexes_q_train = df_indexes_q.query("not test")
Zy_supervised_train_1 = Zy_test[df_indexes_q_train.repetition_1.values]  # 500 x 128
Zy_supervised_train_2 = Zy_test[df_indexes_q_train.repetition_2.values]  # 500 x 128
Zy_supervised_train_3 = Zy_test[df_indexes_q_train.repetition_3.values]  # 500 x 128

# Train an MCCA
mcca = MCCA(n_components=n_components)
mcca.fit(
    [
        Zx_supervised_train_1,
        Zx_supervised_train_2,
        Zx_supervised_train_3,
        Zy_supervised_train_1,
        Zy_supervised_train_2,
        Zy_supervised_train_3,
    ]
)

# Distill each 3 view MCCA into a single linear projector from Z -> Z_mcca (ridge regression).
W_p, b_p = distill_mcca_projector(
    X_train=np.concatenate(
        [Zx_supervised_train_1, Zx_supervised_train_2, Zx_supervised_train_3], axis=0
    ),
    mcca=mcca,
    views=(0, 1, 2),
)

W_q, b_q = distill_mcca_projector(
    X_train=np.concatenate(
        [Zy_supervised_train_1, Zy_supervised_train_2, Zy_supervised_train_3], axis=0
    ),
    mcca=mcca,
    views=(3, 4, 5),
)


for evaluation_set in ["train", "test"]:

    print(f"\nEvaluating on {evaluation_set} set...")
    query = "not test" if evaluation_set == "train" else "test"

    df_indexes_p_test = df_indexes_p.query(query)
    df_indexes_q_test = df_indexes_q.query(query)

    for rep_p in range(1, 4):
        for rep_q in range(1, 4):
            Zx_test_rep = Zx_test[
                df_indexes_p_test[f"repetition_{rep_p}"].values
            ].numpy()
            Zy_test_rep = Zy_test[
                df_indexes_q_test[f"repetition_{rep_q}"].values
            ].numpy()

            # Sanity check: retrieval fails when repetitions don't match
            # np.random.shuffle(Zy_test_rep) # mean rank should be ~251

            Zx_mcca_test = Zx_test_rep @ W_p + b_p
            Zy_mcca_test = Zy_test_rep @ W_q + b_q

            recall, cosine, mean_rank, paired_index = evaluate_retrieval(
                Zx_mcca_test, Zy_mcca_test, device="cuda"
            )
            print(
                f"Test set (rep {rep_p} (subj-01) vs rep {rep_q} (subj-02)) - R@1: {recall:.4f}, Cosine: {cosine:.4f}, Mean Rank: {mean_rank:.2f}"
            )


Evaluating on train set...
Test set (rep 1 (subj-01) vs rep 1 (subj-02)) - R@1: 0.9760, Cosine: 0.5151, Mean Rank: 1.09
Test set (rep 1 (subj-01) vs rep 2 (subj-02)) - R@1: 0.9680, Cosine: 0.4970, Mean Rank: 1.23
Test set (rep 1 (subj-01) vs rep 3 (subj-02)) - R@1: 0.9660, Cosine: 0.5004, Mean Rank: 1.09
Test set (rep 2 (subj-01) vs rep 1 (subj-02)) - R@1: 0.9800, Cosine: 0.5042, Mean Rank: 1.04
Test set (rep 2 (subj-01) vs rep 2 (subj-02)) - R@1: 0.9600, Cosine: 0.4996, Mean Rank: 1.17
Test set (rep 2 (subj-01) vs rep 3 (subj-02)) - R@1: 0.9740, Cosine: 0.4967, Mean Rank: 1.07
Test set (rep 3 (subj-01) vs rep 1 (subj-02)) - R@1: 0.9660, Cosine: 0.5061, Mean Rank: 1.15
Test set (rep 3 (subj-01) vs rep 2 (subj-02)) - R@1: 0.9680, Cosine: 0.4964, Mean Rank: 1.40
Test set (rep 3 (subj-01) vs rep 3 (subj-02)) - R@1: 0.9500, Cosine: 0.5038, Mean Rank: 1.20

Evaluating on test set...
Test set (rep 1 (subj-01) vs rep 1 (subj-02)) - R@1: 0.9160, Cosine: 0.4535, Mean Rank: 1.25
Test set (rep 1